In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ai-tudy/pipeline_detected_family_image

Mounted at /content/drive
/content/drive/MyDrive/ai-tudy/pipeline_detected_family_image


In [ ]:
import sys
sys.path.insert(0, './utils/')
from dataset import ConvDataset
from trainer import train_model, get_y_true_pred
from other_utils import get_model_resnet18, view_classification_report, load_model, save_json
from downloading_from_blur_dataset import download_images_blur_dataset
from PIL import Image
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import os
from tqdm import tqdm
from copy import deepcopy
from sklearn.metrics import f1_score, classification_report




Скачивание изображений с датасета blur-dataset

In [ ]:
download_images_blur_dataset()

Основная работа по обучении модели.

In [ ]:
root_dir_train = 'data/train/blur_dataset'
root_dir_val = 'data/val/blur_dataset'
dataset_train = ConvDataset(root_dir_train)
dataset_val = ConvDataset(root_dir_val)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(dataset_train, batch_size=8, shuffle=True, num_workers=2, pin_memory=True, prefetch_factor=2)
val_loader = DataLoader(dataset_val, batch_size=8, shuffle=False, num_workers=2, pin_memory=True, prefetch_factor=2)

In [ ]:
model = get_model_resnet18().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=7,
    eta_min=1e-6
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 203MB/s]


In [ ]:
path_save = 'models/best_model_blur_Adam.pth'
history = train_model(model, train_loader, val_loader, path_save, criterion, optimizer, scheduler, device=device)



Epoch 1/15


Test: 100%|██████████| 25/25 [00:17<00:00,  1.45it/s]


Train Loss: 0.5150 | Train Acc: 0.8160
Val   Loss: 0.3957 | Val   Acc: 0.8600
--------------------
--------------------

Epoch 2/15


Test: 100%|██████████| 25/25 [00:07<00:00,  3.16it/s]


Train Loss: 0.2989 | Train Acc: 0.8880
Val   Loss: 0.2346 | Val   Acc: 0.9050
--------------------
--------------------

Epoch 3/15


...

++++++++++++++++++++++++++++
Early Stopping: 3 / 5 эпох без улучшений. (Лучший лосс: 0.10495421543717384, Текущий лосс: 0.1905434449017048)
++++++++++++++++++++++++++++
Train Loss: 0.1316 | Train Acc: 0.9580
Val   Loss: 0.1905 | Val   Acc: 0.9150
--------------------
--------------------

Epoch 12/15


Test: 100%|██████████| 25/25 [00:09<00:00,  2.60it/s]


++++++++++++++++++++++++++++
Early Stopping: 4 / 5 эпох без улучшений. (Лучший лосс: 0.10495421543717384, Текущий лосс: 0.19963521346449853)
++++++++++++++++++++++++++++
Train Loss: 0.2525 | Train Acc: 0.9180
Val   Loss: 0.1996 | Val   Acc: 0.9400
--------------------
--------------------

Epoch 13/15


Test: 100%|██████████| 25/25 [00:09<00:00,  2.62it/s]


++++++++++++++++++++++++++++
Early Stopping: 5 / 5 эпох без улучшений. (Лучший лосс: 0.10495421543717384, Текущий лосс: 0.17646415118128062)
++++++++++++++++++++++++++++
Train Loss: 0.3010 | Train Acc: 0.8860
Val   Loss: 0.1765 | Val   Acc: 0.9250
--------------------
Сработала рання остановка!


Тестирование

In [ ]:
model = load_model('models/blur_models/best_model_blur_Adam.pth')

In [ ]:
root_dir_test = 'data/test/blur_dataset'
dataset_test = ConvDataset(root_dir_test)
test_loader = DataLoader(dataset_test, batch_size=4, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
y_true, y_pred = get_y_true_pred(model, test_loader, device)

Test: 100%|██████████| 20/20 [00:27<00:00,  1.35s/it]


In [ ]:
view_classification_report(y_true, y_pred, dataset_test.class_name_list)

              precision    recall  f1-score   support

        blur       0.91      0.80      0.85        40
       sharp       0.82      0.93      0.87        40

    accuracy                           0.86        80
   macro avg       0.87      0.86      0.86        80
weighted avg       0.87      0.86      0.86        80



In [ ]:
save_json(dataset_test.dict_class_label, 'labels/label_blur.json')

JSON успешно сохранен по пути: labels/label_blur.json
